# Tuning de Hiperparámetros con Optuna — Modelo Deep Neuro-Fuzzy

Notebook de búsqueda de hiperparámetros óptimos para el modelo DNF (Deep Neuro-Fuzzy) usando Optuna.

**Contenido del notebook:**
- **Librerías y setup** — Importación de dependencias y configuración de semilla / determinismo.
- **Hiperparámetros a evaluar con Optuna** — Definición del espacio de búsqueda, ejecución del estudio y análisis de resultados.

# Librerías y setup

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
from numpy import dtype
import itertools
from torch.nn.utils import clip_grad_norm_
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, f1_score, precision_recall_curve, accuracy_score
import torch.optim as optim
from typing import Optional, Tuple, Dict, Any, List, Sequence
from collections import Counter, defaultdict
import copy
import os
import pandas as pd
import random
import json
import ast
import yaml
import pickle
import shutil
import optuna
import warnings
import logging

import sys
from pathlib import Path

ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.utils.common import seed_everything as src_seed_everything
from src.data_processing.preprocess import run_data_processing_pipeline as src_run_data_processing_pipeline
from src.training.trainer import run_train_pipeline as src_run_train_pipeline
from src.utils.logging import get_logger as src_get_logger

In [2]:
# ==============================================================================
# CONFIGURACION: Carga desde config.yaml + Determinismo robusto para Optuna
# ==============================================================================

logger = src_get_logger(__name__)
warnings.filterwarnings("ignore")

_CFG_PATH = "../config/config.yaml"

with open(_CFG_PATH, "r", encoding="utf-8") as _f:
    _yaml_cfg = yaml.safe_load(_f)

BASE_SEED = int(_yaml_cfg["project"]["seed"])
_DETERMINISM_CONFIGURED = False

def configure_determinism_runtime_once(seed: int) -> None:
    global _DETERMINISM_CONFIGURED
    if _DETERMINISM_CONFIGURED:
        return
    src_seed_everything(
        seed=seed,
        deterministic_strict=bool(_yaml_cfg["project"].get("deterministic_strict", True)),
        deterministic_warn_only=bool(_yaml_cfg["project"].get("deterministic_warn_only", False)),
    )
    _DETERMINISM_CONFIGURED = True

def reseed_trial(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

configure_determinism_runtime_once(BASE_SEED)

# Hiperparametros a evaluar con Optuna

In [3]:
# ==============================================================================
# CONFIGURACIÓN BASE DEL MODELO — sincronizada con config.yaml
# ==============================================================================

_data_cfg = _yaml_cfg["data_processing"]
_model_cfg = _yaml_cfg["model"]
_train_cfg = _yaml_cfg["training"]

model_cfg = {
    "data_processing": {
        # Ruta a los splits preprocesados (train.csv / val.csv / test.csv)
        "data_path": "../data/splits",
        "target_column": _data_cfg["target_column"],
        "id_column": _data_cfg["id_column"],
        "sequence_length": _data_cfg["sequence_length"],
        "solapamiento_beta": _data_cfg["solapamiento_beta"],
        "fuzzy_processing": copy.deepcopy(_data_cfg["fuzzy_processing"]),
    },
    "lstm": copy.deepcopy(_model_cfg["lstm"]),
    "fuzzy": copy.deepcopy(_model_cfg["fuzzy"]),
    "training_kwargs": {
        "shuffle_train": bool(_train_cfg.get("shuffle_train", False)),
        "batch_size": _train_cfg["batch_size"],
        "lr_rate": _train_cfg["lr_rate"],
        "warmup_epochs": int(_train_cfg.get("warmup_epochs", 0)),
        "weight_decay": float(_train_cfg.get("weight_decay", 1e-5)),
        "num_epochs": _train_cfg["num_epochs"],
        "patience": _train_cfg["patience"],
        "grad_clip": _train_cfg["grad_clip"],
        "scheduler": copy.deepcopy(_train_cfg["scheduler"]),
        "monitor_metric": _train_cfg["monitor_metric"],
        "dnf_loss": copy.deepcopy(_train_cfg["dnf_loss"]),
    },
}

# Compatibilidad con las funciones ya definidas en este notebook, que consumen
# estos campos al nivel superior de data_cfg.
model_cfg["data_processing"]["stats_creation"] = copy.deepcopy(
    model_cfg["data_processing"]["fuzzy_processing"]["stats_creation"]
)

print("model_cfg sincronizado con config.yaml:")
for k, v in model_cfg.items():
    if isinstance(v, dict):
        print(f"  [{k}]")
        for kk, vv in v.items():
            if not isinstance(vv, dict):
                print(f"    {kk}: {vv}")
    else:
        print(f"  {k}: {v}")

model_cfg sincronizado con config.yaml:
  [data_processing]
    data_path: ../data/splits
    target_column: fault_name
    id_column: cycle_id
    sequence_length: 240
    solapamiento_beta: 0.5
    stats_creation: ['mean', 'std', 'min', 'max', 'slope']
  [lstm]
    hidden_size: 64
    num_layers: 1
    dropout: 0.25
    bidirectional: False
    embedding_dim: 32
  [fuzzy]
    n_mf: 3
    n_rules: 48
    train_membership_params: True
    train_rule_params: True
    temperature: 0.7
    t_norm: product
    normalize_rules: True
    init_alpha: sparse
    use_log_bias: False
    lambda_anomaly: 0.8
  [training_kwargs]
    shuffle_train: False
    batch_size: 128
    lr_rate: 0.002
    warmup_epochs: 10
    weight_decay: 1e-05
    num_epochs: 200
    patience: 20
    grad_clip: 1.0
    monitor_metric: 0.50*val_anomaly_f1 + 0.35*fuzzy_f1 - 0.05*dead_rules_ratio - 0.05*alpha_entropy_mean - 0.05*rule_corr


In [4]:
# Base config robusta
if 'model_cfg' in globals() and isinstance(model_cfg, dict):
    tuning_base_cfg = copy.deepcopy(model_cfg)
else:
    raise RuntimeError(
        "No se encontró configuración base para tuning. Ejecuta primero la celda de configuración base."
    )

def _sync_data_cfg(data_cfg):
    fuzzy_processing = data_cfg.setdefault('fuzzy_processing', {})
    data_cfg['stats_creation'] = copy.deepcopy(fuzzy_processing.get('stats_creation'))

    return data_cfg

def _make_stats_creation_variant(*, config_default=True, global_5=False, global_7=False):
    if config_default:
        return ['mean', 'std', 'min', 'max', 'slope']
    if global_5:
        return ['mean', 'std', 'range', 'diff_mean', 'diff_std']
    if global_7:
        return ['mean', 'std', 'min', 'max', 'slope', 'range', 'diff_mean']

    

STATS_CREATION_PRESETS = {
    'config_default': _make_stats_creation_variant(
        config_default=True,
        global_5=False,
        global_7=False,
    ),
    'global_5': _make_stats_creation_variant(
        config_default=False,
        global_5=True,
        global_7=False,
    ),
    'global_7': _make_stats_creation_variant(
        config_default=False,
        global_5=False,
        global_7=True,
    ),
}

def build_trial_cfg(base_cfg, trial):
    cfg = copy.deepcopy(base_cfg)

    cfg.setdefault('data_processing', {})
    cfg.setdefault('lstm', {})
    cfg.setdefault('fuzzy', {})
    cfg.setdefault('training_kwargs', {})
    cfg['training_kwargs'].setdefault('dnf_loss', {})
    cfg['training_kwargs'].setdefault('scheduler', {})


    data_cfg = cfg['data_processing']
    lstm = cfg['lstm']
    fuzzy = cfg['fuzzy']
    train = cfg['training_kwargs']
    dnf = train['dnf_loss']
    scheduler = train['scheduler']

    # -------------------------
    # DATA
    # -------------------------
    data_cfg['sequence_length'] = trial.suggest_categorical('sequence_length', [180, 210, 240])
    data_cfg['solapamiento_beta'] = trial.suggest_categorical('solapamiento_beta', [0.0, 0.5])
    stats_creation_key = trial.suggest_categorical(
        'stats_creation',
        list(STATS_CREATION_PRESETS.keys()),
    )
    data_cfg['fuzzy_processing']['stats_creation'] = copy.deepcopy(
        STATS_CREATION_PRESETS[stats_creation_key]
    )
    _sync_data_cfg(data_cfg)

    # -------------------------
    # LSTM
    # -------------------------
    lstm['hidden_size'] = trial.suggest_categorical('hidden_size', [32, 64, 128])
    lstm['num_layers'] = trial.suggest_categorical('num_layers', [1, 2])
    lstm['dropout'] = trial.suggest_float('dropout', 0.10, 0.35, step=0.05)
    lstm['bidirectional'] = trial.suggest_categorical('bidirectional', [False, True])
    lstm['embedding_dim'] = trial.suggest_categorical('embedding_dim', [16, 32, 64])

    # -------------------------
    # Fuzzy
    # -------------------------
    fuzzy['n_mf'] = trial.suggest_int('n_mf', 3, 5, step=1)
    fuzzy['n_rules'] = trial.suggest_categorical('n_rules', [24, 48, 64, 96])

    fuzzy['temperature'] = trial.suggest_float('temperature', 0.7, 1.1, step=0.05)
    fuzzy['t_norm'] = trial.suggest_categorical('t_norm', ['product', 'min'])
    fuzzy['normalize_rules'] = trial.suggest_categorical('normalize_rules', [True, False])
    fuzzy['init_alpha'] = trial.suggest_categorical('init_alpha', ['sparse', 'random'])
    fuzzy['lambda_anomaly'] = round(trial.suggest_float('lambda_anomaly', 0.7, 0.90, step=0.05), 2)

    # -------------------------
    # Entrenamiento
    # -------------------------
    train['batch_size'] = trial.suggest_categorical('batch_size', [64, 128, 256])
    train['lr_rate'] = trial.suggest_float('lr_rate', 2.5e-4, 2.5e-3, step=2.5e-4)

    # Scheduler: solo el tipo; el resto queda sincronizado con config.yaml
    scheduler['type'] = trial.suggest_categorical(
        'scheduler_type',
        ['plateau', 'cosine_warmup', 'cosine'],
    )

    # -------------------------
    # DNF Loss
    # -------------------------
    dnf['anomaly_weight'] = trial.suggest_float('anomaly_weight', 1.7, 2.50, step=0.10)
    dnf['aux_anom_dl_weight'] = trial.suggest_float('aux_anom_dl_weight', 0.30, 0.60, step=0.05)
    dnf['aux_anom_fuzzy_weight'] = trial.suggest_float('aux_anom_fuzzy_weight', 0.75, 1.25, step=0.05)
    dnf['diversity_weight'] = trial.suggest_float('diversity_weight', 0.0, 0.02, step=0.005)
    dnf['alpha_entropy_weight'] = trial.suggest_float('alpha_entropy_weight', 0.005, 0.020, step=0.005)
    dnf['rule_structure_div_weight'] = trial.suggest_float('rule_structure_div_weight', 0.00, 0.05, step=0.01)
    dnf['rule_usage_balance_weight'] = trial.suggest_float('rule_usage_balance_weight', 0.00, 0.05, step=0.01)

    return cfg

def select_best_epoch_from_history(history):
    scores = np.asarray(history.get("monitor_score", []), dtype=float)
    if scores.size == 0:
        raise ValueError("History vacío: falta 'monitor_score'.")

    best_epoch_idx = int(np.nanargmax(scores))
    best_score = float(scores[best_epoch_idx])

    best_metrics = {
        "best_epoch": best_epoch_idx + 1,
        "score": best_score,
        "val_anomaly_f1": float(np.asarray(history["val_anomaly_f1"], dtype=float)[best_epoch_idx]),
        "val_anomaly_auc": float(np.asarray(history["val_anomaly_auc"], dtype=float)[best_epoch_idx]),
        "fuzzy_f1": float(np.asarray(history["val_fuzzy_f1"], dtype=float)[best_epoch_idx]),
        "dl_f1": float(np.asarray(history["val_dl_f1"], dtype=float)[best_epoch_idx]),
        "alpha_entropy_mean": float(np.asarray(history["val_alpha_entropy"], dtype=float)[best_epoch_idx]),
        "dead_rules": int(np.asarray(history["val_dead_rules"], dtype=float)[best_epoch_idx]),
        "rule_corr_mean": float(np.asarray(history["val_rule_corr"], dtype=float)[best_epoch_idx]),
    }
    return best_score, best_metrics


def _build_data_cfg_for_trial(cfg):
    data_cfg_global = globals().get('data_cfg')
    if isinstance(data_cfg_global, dict):
        data_cfg_local = copy.deepcopy(data_cfg_global)
    else:
        data_cfg_local = {}

    data_cfg_local.update(copy.deepcopy(cfg.get('data_processing', {})))

    # Garantiza que data_path apunte a los splits procesados
    if 'data_path' not in data_cfg_local:
        data_cfg_local['data_path'] = '../data/splits'

    if 'sequence_length' not in data_cfg_local:
        raise RuntimeError("No se pudo resolver 'sequence_length' en data_cfg para el trial.")

    # Asegura normal_tokens requerido por run_data_processing_pipeline
    if data_cfg_local.get('normal_tokens') is None:
        if '_yaml_cfg' in globals() and isinstance(_yaml_cfg, dict):
            data_cfg_local['normal_tokens'] = copy.deepcopy(
                _yaml_cfg.get('data_processing', {}).get('normal_tokens',{})
            )
        elif 'NORMAL_TOKENS' in globals() and NORMAL_TOKENS is not None:
            data_cfg_local['normal_tokens'] = sorted(list(NORMAL_TOKENS))

    if data_cfg_local.get('normal_tokens') is None or len(data_cfg_local.get('normal_tokens', [])) == 0:
        raise RuntimeError(
            "No se pudo resolver 'normal_tokens' para el trial. "
            "Verifica data_processing.normal_tokens (o data.normal_tokens) en config.yaml."
        )

    return data_cfg_local

_PREPROCESS_CACHE = {}

def _make_preprocess_cache_key(data_cfg_trial):
    stats_creation = tuple(data_cfg_trial['fuzzy_processing']['stats_creation'])
    return (
        data_cfg_trial.get('data_path'),
        data_cfg_trial.get('target_column'),
        data_cfg_trial.get('id_column'),
        int(data_cfg_trial.get('sequence_length')),
        float(data_cfg_trial.get('solapamiento_beta')),
        stats_creation,
    )

def _resolve_split_data_for_trial(cfg):
    if 'src_run_data_processing_pipeline' not in globals():
        raise RuntimeError("No se encontró src_run_data_processing_pipeline en memoria para recalcular split_data por trial.")

    data_cfg_trial = _build_data_cfg_for_trial(cfg)
    cache_key = _make_preprocess_cache_key(data_cfg_trial)

    if cache_key not in _PREPROCESS_CACHE:
        split_local, stats_local, scaler_x_local, scaler_num_local = src_run_data_processing_pipeline(data_cfg_trial, config=_yaml_cfg)
        _PREPROCESS_CACHE[cache_key] = (split_local, stats_local, scaler_x_local, scaler_num_local)

    split_local, stats_local, scaler_x_local, scaler_num_local = _PREPROCESS_CACHE[cache_key]
    return copy.deepcopy(split_local), copy.deepcopy(stats_local), copy.deepcopy(scaler_x_local), copy.deepcopy(scaler_num_local)

TEMP_OPTUNA_DIR = "../models/artifacts/temp_optuna_trials"
TEMP_BEST_BUNDLE_DIR = os.path.join(TEMP_OPTUNA_DIR, "temp_best")


def _jsonable(obj):
    if obj is None or isinstance(obj, (str, bool, int)):
        return obj

    if isinstance(obj, float):
        return obj if np.isfinite(obj) else None

    if isinstance(obj, np.integer):
        return int(obj)

    if isinstance(obj, np.floating):
        value = float(obj)
        return value if np.isfinite(value) else None

    if isinstance(obj, np.ndarray):
        return _jsonable(obj.tolist())

    if isinstance(obj, (list, tuple)):
        return [_jsonable(v) for v in obj]

    if isinstance(obj, dict):
        return {str(k): _jsonable(v) for k, v in obj.items()}

    return str(obj)

def deep_update(dst, src):
    for key, value in src.items():
        if isinstance(value, dict) and isinstance(dst.get(key), dict):
            deep_update(dst[key], value)
        else:
            dst[key] = copy.deepcopy(value)
    return dst


def to_builtin_yaml(obj):
    if isinstance(obj, dict):
        return {str(k): to_builtin_yaml(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [to_builtin_yaml(v) for v in obj]
    if isinstance(obj, tuple):
        return [to_builtin_yaml(v) for v in obj]
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj

def save_best_trial_bundle(
    *,
    split_data_local,
    split_data_df_stats_local,
    scaler_x_local,
    scaler_num_local,
    train_out,
    cfg,
    trial,
    score,
    best_metrics,
    checkpoint_path,
    bundle_dir=TEMP_BEST_BUNDLE_DIR,
):
    staging_dir = f"{bundle_dir}_staging"

    if os.path.exists(staging_dir):
        shutil.rmtree(staging_dir)

    os.makedirs(staging_dir, exist_ok=True)

    with open(os.path.join(staging_dir, "scaler.pkl"), "wb") as f:
        pickle.dump(
            {
                "scaler_x": scaler_x_local,
                "scaler_num": scaler_num_local,
            },
            f,
        )

    for split_name in ("train", "val", "test"):
        np.save(
            os.path.join(staging_dir, f"X_{split_name}.npy"),
            split_data_local["X"][split_name],
        )
        np.save(
            os.path.join(staging_dir, f"y_{split_name}.npy"),
            split_data_local["y"][split_name],
        )
        split_data_df_stats_local[split_name].to_csv(
            os.path.join(staging_dir, f"stats_{split_name}.csv"),
            index=False,
        )

    history = train_out["history"]
    with open(os.path.join(staging_dir, "training_history.json"), "w", encoding="utf-8") as f:
        json.dump(_jsonable(history), f, indent=2, ensure_ascii=False, allow_nan=False)

    best_threshold = train_out.get("best_threshold", 0.5)
    best_epoch = train_out.get("best_epoch", best_metrics.get("best_epoch"))
    n_epochs_trained = len(history.get("monitor_score", []))

    meta = {
        "best_threshold": float(best_threshold),
        "best_epoch": int(best_epoch),
        "n_epochs_trained": int(n_epochs_trained),
        "trial_number": int(trial.number),
        "params": dict(trial.params),
        "score": float(score),
        "best_metrics": best_metrics,
        "checkpoint_path": checkpoint_path,
        "model_cfg": train_out.get("model_cfg"),
    }

    with open(os.path.join(staging_dir, "best_meta.json"), "w", encoding="utf-8") as f:
        json.dump(_jsonable(meta), f, indent=2, ensure_ascii=False, allow_nan=False)

    if os.path.exists(bundle_dir):
        shutil.rmtree(bundle_dir)

    shutil.move(staging_dir, bundle_dir)

    return {
        "bundle_dir": bundle_dir,
        "best_meta_path": os.path.join(bundle_dir, "best_meta.json"),
    }

_CURRENT_BEST_SCORE = -np.inf
_CURRENT_BEST_CKPT = None
def objective(trial):
    global _CURRENT_BEST_SCORE, _CURRENT_BEST_CKPT

    if optuna is None:
        raise ModuleNotFoundError("Optuna no está instalado en este kernel. Instala 'optuna' para ejecutar el tuning.")
    
    # Determinismo estricto: misma semilla + comportamiento determinista en PyTorch
    reseed_trial(BASE_SEED)

    total_trials = globals().get('N_TRIALS') 
    
    print("\n" + "=" * 100)
    print(f"INICIANDO TRIAL {trial.number + 1} de {total_trials}                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             ")
    print("=" * 100)

    cfg = build_trial_cfg(base_cfg=tuning_base_cfg, trial=trial)
    save_prefix = f"optuna_trial_{trial.number}"
    ckpt_path = os.path.join(TEMP_OPTUNA_DIR, f"best_{save_prefix}.pt")

    split_data_local, split_data_df_stats_local, scaler_x_local, scaler_num_local = _resolve_split_data_for_trial(cfg)

    try:
        data_cfg_model = copy.deepcopy(_yaml_cfg["data_processing"])
        trial_data = cfg["data_processing"]

        data_cfg_model["sequence_length"] = trial_data["sequence_length"]
        data_cfg_model["solapamiento_beta"] = trial_data["solapamiento_beta"]
        data_cfg_model["fuzzy_processing"] = copy.deepcopy(
            trial_data["fuzzy_processing"]
        )

        model_cfg_for_train = {
            "seed": int(_yaml_cfg.get("project", {}).get("seed", 42)),
            "data_processing": data_cfg_model,
            "lstm": copy.deepcopy(cfg["lstm"]),
            "fuzzy": copy.deepcopy(cfg["fuzzy"]),
            "training_kwargs": copy.deepcopy(cfg["training_kwargs"]),
        }
        
        train_out = src_run_train_pipeline(
            split_data=split_data_local,
            model_cfg=model_cfg_for_train,
            split_data_df_stats=split_data_df_stats_local,
            save_prefix=save_prefix,
            save_dir=TEMP_OPTUNA_DIR,
        )

        history = train_out['history']
        score, best_metrics = select_best_epoch_from_history(history)

        trial.report(score, step=int(best_metrics['best_epoch']))

        for k, v in best_metrics.items():
            trial.set_user_attr(k, v)

        trial.set_user_attr('best_threshold', float(train_out.get('best_threshold')))
        trial.set_user_attr('best_epoch', int(train_out.get('best_epoch')))
        trial.set_user_attr('n_epochs_trained', int(len(history.get('monitor_score', []))))

        if trial.should_prune():
            raise optuna.TrialPruned()

         # Si este trial mejora al mejor histórico de esta ejecución, conservar su checkpoint
        trial.set_user_attr("checkpoint_path", ckpt_path)
        if np.isfinite(score) and float(score) > _CURRENT_BEST_SCORE:
            prev_ckpt = _CURRENT_BEST_CKPT

            bundle_paths = save_best_trial_bundle(
                split_data_local=split_data_local,
                split_data_df_stats_local=split_data_df_stats_local,
                scaler_x_local=scaler_x_local,
                scaler_num_local=scaler_num_local,
                train_out=train_out,
                cfg=cfg,
                trial=trial,
                score=score,
                best_metrics=best_metrics,
                checkpoint_path=ckpt_path,
            )

            _CURRENT_BEST_SCORE = float(score)
            _CURRENT_BEST_CKPT = ckpt_path

            trial.set_user_attr("bundle_dir", bundle_paths["bundle_dir"])
            trial.set_user_attr("best_meta_path", bundle_paths["best_meta_path"])

            if prev_ckpt and prev_ckpt != ckpt_path and os.path.exists(prev_ckpt):
                try:
                    os.remove(prev_ckpt)
                except Exception:
                    pass
        return score

    except RuntimeError as exc:
        message = str(exc).lower()
        if 'out of memory' in message or ('cuda' in message and 'memory' in message):
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            raise optuna.TrialPruned() from exc
        raise
    finally:
        
        # Borrar solo si NO es el mejor actual
        if ckpt_path != _CURRENT_BEST_CKPT and os.path.exists(ckpt_path):
            try:
                os.remove(ckpt_path)
            except Exception:
                pass

In [5]:
# ==============================================================================
# EJECUCIÓN DEL ESTUDIO DE OPTUNA
# ==============================================================================

N_TRIALS = 80

# Configuración del sampler y el pruner
sampler = optuna.samplers.TPESampler(
    seed=BASE_SEED,
    multivariate=True,
    group=True,
)
pruner = optuna.pruners.MedianPruner(
    n_startup_trials=8,
    n_warmup_steps=3,
    interval_steps=1,
    n_min_trials=8,
)

# Creación del estudio
study = optuna.create_study(
    direction="maximize",
    sampler=sampler,
    pruner=pruner,
    study_name="parallel_dnf_tuning",
    load_if_exists=True,
)

study.optimize(
    objective,
    n_trials=N_TRIALS,
    gc_after_trial=True,
    show_progress_bar=True,
)

# ========================================
# PROCESAMIENTO DE RESULTADOS FINALES
# ========================================

best_trial = study.best_trial
print(f"Mejor trial: {best_trial.number}")
print(f"Mejor score: {best_trial.value:.6f}")

print("\n=== Mejores hiperparámetros optimizados ===")
print(json.dumps(best_trial.params, indent=2, ensure_ascii=False))

BEST_MODEL_PATH = "../models/artifacts/best_dnf_model.pt"
OPTUNA_SCALER_PATH = "../models/artifacts/scaler.pkl"
OPTUNA_RESULTS_PATH = "../models/metrics/results.json"
OPTUNA_HISTORY_PATH = "../models/metrics/training_history.json"
OPTUNA_BEST_TRIAL_PATH = "../models/metrics/best_optuna_trial.json"
PROCESSED_DIR = "../data/processed"

BEST_META_PATH = os.path.join(TEMP_BEST_BUNDLE_DIR, "best_meta.json")

if not os.path.exists(BEST_META_PATH):
    raise FileNotFoundError(
        f"No existe el bundle del mejor trial en {BEST_META_PATH}. "
        "El estudio no puede finalizar sin reentreno porque no hay artifacts temporales guardados."
    )

with open(BEST_META_PATH, "r", encoding="utf-8") as f:
    best_meta = json.load(f)

best_cfg = copy.deepcopy(best_meta.get("model_cfg", {}))

if int(best_meta["trial_number"]) != int(best_trial.number):
    raise RuntimeError(
        f"El bundle temporal pertenece al trial {best_meta['trial_number']}, "
        f"pero study.best_trial es el trial {best_trial.number}. "
        "Revisa si reutilizaste un estudio anterior sin conservar su temp_best."
    )

os.makedirs("../models/artifacts", exist_ok=True)
os.makedirs("../models/metrics", exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

best_ckpt_src = best_meta["checkpoint_path"]
if not os.path.exists(best_ckpt_src):
    raise FileNotFoundError(f"No existe el checkpoint del mejor trial: {best_ckpt_src}")

ckpt_obj = torch.load(best_ckpt_src, map_location="cpu")

if isinstance(ckpt_obj, dict) and "model_state_dict" in ckpt_obj:
    checkpoint_payload = dict(ckpt_obj)
elif isinstance(ckpt_obj, dict) and "state_dict" in ckpt_obj:
    checkpoint_payload = dict(ckpt_obj)
    checkpoint_payload["model_state_dict"] = checkpoint_payload.pop("state_dict")
else:
    checkpoint_payload = {
        "model_state_dict": ckpt_obj,
    }

checkpoint_payload.update(
    {
        "model_cfg": best_cfg,
        "best_threshold": float(best_meta["best_threshold"]),
        "best_epoch": int(best_meta["best_epoch"]),
        "n_epochs_trained": int(best_meta["n_epochs_trained"]),
        "params": best_meta["params"],
    }
)

torch.save(checkpoint_payload, BEST_MODEL_PATH)

shutil.copy2(
    os.path.join(TEMP_BEST_BUNDLE_DIR, "scaler.pkl"),
    OPTUNA_SCALER_PATH,
)

for split_name in ("train", "val", "test"):
    shutil.copy2(
        os.path.join(TEMP_BEST_BUNDLE_DIR, f"X_{split_name}.npy"),
        os.path.join(PROCESSED_DIR, f"X_{split_name}.npy"),
    )
    shutil.copy2(
        os.path.join(TEMP_BEST_BUNDLE_DIR, f"y_{split_name}.npy"),
        os.path.join(PROCESSED_DIR, f"y_{split_name}.npy"),
    )
    shutil.copy2(
        os.path.join(TEMP_BEST_BUNDLE_DIR, f"stats_{split_name}.csv"),
        os.path.join(PROCESSED_DIR, f"stats_{split_name}.csv"),
    )

shutil.copy2(
    os.path.join(TEMP_BEST_BUNDLE_DIR, "training_history.json"),
    OPTUNA_HISTORY_PATH,
)

with open(OPTUNA_RESULTS_PATH, "w", encoding="utf-8") as f:
    json.dump(
        {
            "best_threshold": float(best_meta["best_threshold"]),
            "best_epoch": int(best_meta["best_epoch"]),
            "n_epochs_trained": int(best_meta["n_epochs_trained"]),
        },
        f,
        indent=2,
        ensure_ascii=False,
    )

best_results = {
    "best_trial_number": int(best_trial.number),
    "best_value": float(best_trial.value),
    "best_threshold": float(best_meta["best_threshold"]),
    "best_epoch": int(best_meta["best_epoch"]),
    "n_epochs_trained": int(best_meta["n_epochs_trained"]),
    "model_path": BEST_MODEL_PATH,
    "scaler_path": OPTUNA_SCALER_PATH,
    "results_path": OPTUNA_RESULTS_PATH,
    "history_path": OPTUNA_HISTORY_PATH,
    "processed_dir": PROCESSED_DIR,
    "source_checkpoint_path": best_ckpt_src,
    "temp_best_bundle": TEMP_BEST_BUNDLE_DIR,
    "params": best_meta["params"],
    "user_attrs": dict(best_trial.user_attrs),
}

with open(OPTUNA_BEST_TRIAL_PATH, "w", encoding="utf-8") as f:
    json.dump(_jsonable(best_results), f, indent=2, ensure_ascii=False, allow_nan=False)

print(f"\nModelo Optuna guardado en: {BEST_MODEL_PATH}")
print(f"Resultados guardados en: {OPTUNA_RESULTS_PATH}")
print(f"Historial de entrenamiento guardado en: {OPTUNA_HISTORY_PATH}")
print(f"Mejor trial guardado en: {OPTUNA_BEST_TRIAL_PATH}")
print(f"Scalers guardados en: {OPTUNA_SCALER_PATH}")
print(f"Datos procesados guardados en: {PROCESSED_DIR}")

if os.path.exists(TEMP_OPTUNA_DIR):
    try:
        shutil.rmtree(TEMP_OPTUNA_DIR)
        print(f"Carpeta temporal '{TEMP_OPTUNA_DIR}' eliminada con éxito.")
    except Exception as exc:
        print(f"No se pudo eliminar la carpeta temporal de forma automática: {exc}")

trials_df = study.trials_dataframe(attrs=("number", "value", "state", "params", "user_attrs"))
if not trials_df.empty:
    trials_df = trials_df.sort_values(by="value", ascending=False)

    print("\n=== Top 10 trials ===")
    with pd.option_context('display.max_columns', None):
        print(trials_df.head(10))

print("\n=== Importancia de hiperparámetros ===")
try:
    importances = optuna.importance.get_param_importances(study)
    print(json.dumps(importances, indent=2, ensure_ascii=False))
except Exception as exc:
    print(f"No se pudo calcular importancias: {exc}")

[I 2026-07-21 10:44:25,550] A new study created in memory with name: parallel_dnf_tuning


  0%|          | 0/80 [00:00<?, ?it/s]


INICIANDO TRIAL 1 de 80                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             
2026-07-21 10:44:25. Cargando dato